# MIW CWFS Intrinsic Check (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-13
**Last Modified:** 2026-07-13
**Status:** In Progress
**Keywords:** MIW, intrinsic wavefront, CWFS, IntrinsicZernikes, OCS, CCS, AOS validation

## Description

The Measured Intrinsic Wavefront (MIW) went live in the AOS.  This notebook
verifies that the **correct intrinsic values were used** for the 4 corner
wavefront sensors (CWFS) on regular (science) images, by independently
recomputing the expected intrinsic from **my own calibration source tables** and
comparing to the `zk_intrinsic_*` stored in the live aggregate tables.

The live path (traced in `ts_intrinsic_wavefront`): the AOS applies
`lsst.ip.isr.IntrinsicZernikes(table=<per-detector CCS>, table_ocs=<OCS>)` and
stores, per donut,
`calib.getIntrinsicZernikes(field_x, field_y, rotTelPos=...)`.
So the check rebuilds that exact calib object from the OCS + per-detector CCS
source parquets, evaluates it at each CWFS donut's field position and the
visit rotator, and differences against the aggregate.

Because a handful of frame conventions (field-argument order, rotator
sign/offset, and which of the aggregate's OCS/CCS/NW intrinsic columns
`getIntrinsicZernikes` reproduces) are not obvious a priori, the notebook
**auto-solves** that small discrete space against the aggregate: the config that
reproduces the stored intrinsic (correlation ~1, slope ~1, tiny residual)
confirms the correct values were used *and* tells us the convention; if **no**
config matches, that is the alarm (wrong / stale calib ingested, or a bug).

A separate validation section plots the OCS and CCS focal-plane maps (as in
`aos_miw_ocs_ccs_maps.ipynb`) as a further sanity check on the maps behind the
calib.

**Output:** `output/miw_cwfs_intrinsic_check.parquet` (per donut x Noll:
expected vs aggregate + residual) and the comparison / map figures.

**References:** `ts_intrinsic_wavefront` `calib_tables.py`,
`bin.src/run_make_calib_tables.py`, `bin.src/ingest_calib_tables.py`
(`IntrinsicZernikes(table=ccs, table_ocs=ocs)` /
`getIntrinsicZernikes(field_x, field_y, rotTelPos)`); `aos_miw_ocs_ccs_maps.ipynb`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-13 | Aaron Roodman | Initial version — CWFS intrinsic verification vs live aggregate + OCS/CCS map validation |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load calib source tables -> IntrinsicZernikes](#calib)
4. [Validation: OCS / CCS focal-plane maps](#maps)
5. [Load aggregate CWFS donuts (last night)](#agg)
6. [Recompute expected intrinsic + auto-solve convention](#recompute)
7. [Comparison plots + PASS/FAIL](#compare)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
from datetime import date, timedelta

# ---- Butler + the per-donut-pair Raw aggregate for the night's regular images ----
# quickLook is Summit-only; at USDF read the nightlyValidation run for the night.
# Find the run for another DAY_OBS:  query_datasets(dt,
#   collections="LSSTCam/runs/nightlyValidation/*", where="instrument=\'LSSTCam\'
#   AND exposure.day_obs=<d>", find_first=False) -> {r.run}.  (USE_SUMMIT_BUTLER=True
#   uses the Summit embargo quickLook chain instead -- but that has only the Avg.)
USE_SUMMIT_BUTLER = False                        # False -> Butler(BUTLER_REPO, collections=COLLECTIONS)
BUTLER_REPO  = "/repo/embargo"                   # 20260713 AOS output lives here (not /repo/main)
COLLECTIONS  = ["LSSTCam/runs/nightlyValidation/68"]   # the run holding DAY_OBS=20260713
DATASET_TYPE = "aggregateAOSVisitTableRaw"       # nightlyValidation has Raw + Avg + zernikes
DAY_OBS      = 20260713
MAX_SEQ      = None      # None = all visits; or an int to load only the first N seq_num (fast peek)

# ---- My own calib SOURCE tables (mirrors the USDF rubin-data layout) ----
CALIB_DIR = "~/notebooks/rubin-data/aos/intrinsic_zernikes/v2_ajr"   # OCS + per-detector CCS
OCS_FILE  = "intrinsic_aberrations_OCS.parquet" # per-instrument OCS table (whole map)
# per-detector CCS tables expected as  intrinsic_aberrations_CCS_det<NNN>.parquet

# ---- MIW maps for the OCS/CCS validation plots (set None to skip that section) ----
MAPS_PATH = "~/notebooks/rubin-data/aos/intrinsic_zernikes/intrinsic_split_maps.parquet"

# ---- CWFS detectors: the Avg table carries the 4 SW0 corners (as in olr) ----
CWFS = {191: "R00_SW0", 195: "R04_SW0", 199: "R40_SW0", 203: "R44_SW0"}

# ---- frame conventions: SET these, don't auto-solve (a mismatch shows in the plots) ----
FIELD_ORDER   = ("thx_CCS", "thy_CCS")           # (field_x, field_y) base cols for getIntrinsicZernikes
RTP_SIGN      = +1                                # meta rotTelPos (converted to deg) is multiplied by this
COMPARE_FRAME = "OCS"                             # aggregate zk_intrinsic frame to compare: OCS | CCS | NW
TOL_UM        = 0.01                             # PASS if max median|expected-aggregate| <= this (µm)
OUT_PARQUET   = "output/miw_cwfs_intrinsic_check.parquet"
FP_RADIUS_DEG = 1.75
PCT           = 98                               # map color percentile

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from tqdm import tqdm                              # plain text bar (no ipywidgets)

from lsst.daf.butler import Butler
from lsst.ip.isr import IntrinsicZernikes

if USE_SUMMIT_BUTLER:                            # same as olr/nightly_table.py
    from lsst.summit.utils import butlerUtils
    butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)
else:
    butler = Butler(BUTLER_REPO, collections=COLLECTIONS)
NAME2ID = {v: k for k, v in CWFS.items()}


def to_deg(a):
    """Angles to degrees (aggregate positions may be stored in radians)."""
    a = np.asarray(a, float)
    return np.degrees(a) if np.nanmax(np.abs(a)) < 0.2 else a

<a id='calib'></a>
## 3. Load calib source tables -> IntrinsicZernikes

Reconstructs the exact `IntrinsicZernikes(table=ccs, table_ocs=ocs)` the AOS ingested, one per CWFS detector, from **my own** source parquets.

In [ ]:
# Rebuild the exact calib the AOS ingested: shared OCS + per-detector CCS.
# ts_wep evaluates the intrinsic per donut with THAT donut's own CCD calib, so we
# need both the extra (SW0, the row detector) AND the intra (SW1 = SW0+1) halves.
cdir = Path(CALIB_DIR).expanduser()
ocs = Table.read(str(cdir / OCS_FILE), format="parquet")
det_ids = sorted(set(CWFS) | {d + 1 for d in CWFS})   # SW0 (extra) + SW1 (intra)
calib_by_det, missing = {}, []
for det in det_ids:
    p = cdir / f"intrinsic_aberrations_CCS_det{det:03d}.parquet"
    if p.exists():
        calib_by_det[det] = IntrinsicZernikes(table=Table.read(str(p), format="parquet"),
                                              table_ocs=ocs)
    else:
        missing.append(det)
if missing:
    print(f"WARNING: missing CCS tables for detectors {missing} "
          f"(need the full v2_ajr set incl. SW1: 192/196/200/204)")
assert calib_by_det, "no calibs loaded -- check CALIB_DIR / OCS_FILE / CCS filenames"

calib_noll = [int(x) for x in np.asarray(next(iter(calib_by_det.values())).noll_indices)]
print(f"loaded {len(calib_by_det)} calibs (SW0 extra + SW1 intra): {sorted(calib_by_det)}")
print(f"calib Noll (n={len(calib_noll)}): {calib_noll}")

<a id='maps'></a>
## 4. Validation: OCS / CCS focal-plane maps

Reuses `optatmo/code/plot_miw_maps.py` on the calib's own interpolators: Z4 OCS (`interpolator_ocs` on a global grid) vs Z4 CCS (rendered **per-CCD** from each detector's interpolator, so the per-CCD height steps show at full resolution), then OCS maps Z5-Z11. DVCS orientation (x_field vertical, y_field horizontal).

In [ ]:
# Reuse optatmo/code/plot_miw_maps.py: render from the calib's OWN interpolators.
#   Z4 OCS  -> calib.interpolator_ocs on a global grid (imshow)
#   Z4 CCS  -> one patch per CCD via that detector's interpolator (per-CCD height
#              steps preserved, no cross-CCD blending)
#   Z5..Z11 OCS.  DVCS orientation: x_field vertical, y_field horizontal.
import glob
from matplotlib.patches import Circle

LIM, OCS_N, CCS_SUB = FP_RADIUS_DEG, 71, 10

# a calib per available detector -> full-focal-plane CCS map (all CCDs if built)
calibs_all = {}
for f in sorted(glob.glob(str(cdir / "intrinsic_aberrations_CCS_det*.parquet"))):
    det = int(re.search(r"det(\d+)\.parquet", f).group(1))
    calibs_all[det] = IntrinsicZernikes(table=Table.read(f, format="parquet"), table_ocs=ocs)
cal0 = next(iter(calibs_all.values()))
noll_c = [int(j) for j in np.asarray(cal0.noll_indices)]
kc = {j: i for i, j in enumerate(noll_c)}
print(f"maps: {len(calibs_all)} detector calibs; Noll {noll_c}")


def _vlim(v, pct=99.0):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return max(float(np.percentile(np.abs(v), pct)), 1e-6) if v.size else 1.0


def _dvcs(ax, title):
    ax.add_patch(Circle((0, 0), FP_RADIUS_DEG, fill=False, ls="--", color="k", lw=0.5, alpha=0.5))
    ax.set_aspect("equal"); ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM)
    ax.set_title(title, fontsize=9); ax.tick_params(labelsize=6)
    ax.set_xlabel("y_field [deg]", fontsize=7); ax.set_ylabel("x_field [deg]", fontsize=7)


def _imshow(ax, Z, title, sample_xy=None):
    vl = _vlim(Z)
    im = ax.imshow(Z.T, origin="lower", extent=[-LIM, LIM, -LIM, LIM], cmap="RdBu_r",
                   vmin=-vl, vmax=vl, interpolation="nearest")
    if sample_xy is not None:
        ax.plot(sample_xy[1], sample_xy[0], ".", ms=0.5, color="k", alpha=0.15)
    _dvcs(ax, title); return im


_ax = np.linspace(-LIM, LIM, OCS_N)
gx, gy = np.meshgrid(_ax, _ax)
gg = np.column_stack([gx.ravel(), gy.ravel()])
ocs_xy = (np.asarray(cal0.field_x_ocs), np.asarray(cal0.field_y_ocs))
def ocs_map(j):
    return np.asarray(cal0.interpolator_ocs(gg))[:, kc[j]].reshape(gx.shape)

# per-CCD Z4 CCS patches (skip whole-FP height-only field-edge CCDs)
patches, n_skip = [], 0
for d, c in calibs_all.items():
    fx, fy = np.asarray(c.field_x), np.asarray(c.field_y)
    if fx.size >= 0.5 * ocs_xy[0].size:
        n_skip += 1; continue
    ex = np.linspace(fx.min(), fx.max(), CCS_SUB + 1)
    ey = np.linspace(fy.min(), fy.max(), CCS_SUB + 1)
    ggx, ggy = np.meshgrid(0.5 * (ex[:-1] + ex[1:]), 0.5 * (ey[:-1] + ey[1:]))
    z = np.asarray(c.interpolator(np.column_stack([ggx.ravel(), ggy.ravel()])))
    patches.append((ex, ey, z[:, kc[4]].reshape(ggx.shape)))
vl_ccs = _vlim(np.concatenate([p[2].ravel() for p in patches])) if patches else 1.0

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
i0 = _imshow(ax[0], ocs_map(4), "Z4 OCS [µm] (calib.interpolator_ocs)", sample_xy=ocs_xy)
fig.colorbar(i0, ax=ax[0], shrink=0.8)
im = None
for ex, ey, z in patches:                     # DVCS: horizontal=field_y, vertical=field_x
    im = ax[1].pcolormesh(ey, ex, np.ma.masked_invalid(z.T), cmap="RdBu_r",
                          vmin=-vl_ccs, vmax=vl_ccs, shading="flat")
_dvcs(ax[1], f"Z4 CCS [µm] (per-CCD {CCS_SUB}x{CCS_SUB}, incl. height)")
if im is not None:
    fig.colorbar(im, ax=ax[1], shrink=0.8)
fig.suptitle("MIW calib (v2_ajr) — Z4 (defocus): OCS vs CCS  (DVCS)")
fig.tight_layout(); plt.show()
print(f"CCS: {len(patches)} footprint CCDs rendered, {n_skip} height-only skipped")

js = [j for j in range(5, 12) if j in kc]     # OCS maps Z5..Z11
if js:
    cols = 4; rows = int(np.ceil(len(js) / cols))
    fig, axs = plt.subplots(rows, cols, figsize=(4.0 * cols, 3.7 * rows))
    axs = np.atleast_1d(axs).ravel()
    for a, j in zip(axs, js):
        fig.colorbar(_imshow(a, ocs_map(j), f"Z{j} OCS [µm]", sample_xy=ocs_xy), ax=a, shrink=0.8)
    for a in axs[len(js):]:
        a.axis("off")
    fig.suptitle("MIW calib (v2_ajr) — OCS maps Z5-Z11  (DVCS)")
    fig.tight_layout(); plt.show()

<a id='agg'></a>
## 5. Load aggregate CWFS donuts (last night)

The live **Raw** aggregate for regular images (one row per donut pair, so we see the intrinsic used per donut), via the Summit embargo butler keyed by `exposure.day_obs` (same butler as `rubin-work/olr/code/nightly_table.py`, but the Raw dataset rather than Avg). Keep the CWFS rows with their field positions and the stored `zk_intrinsic_*`.

In [ ]:
# Pull last night's per-donut-pair aggregates (Raw) and keep the CWFS rows.
refs = list(butler.query_datasets(DATASET_TYPE,
            where=f"instrument='LSSTCam' AND exposure.day_obs = {DAY_OBS}"))
refs.sort(key=lambda r: int(r.dataId["visit"]))     # by seq_num order
if MAX_SEQ:                                          # fast peek: first N seq_num only
    refs = refs[:int(MAX_SEQ)]
    print(f"MAX_SEQ={MAX_SEQ}: limiting to the first {len(refs)} visits")
print(f"loading {len(refs)} {DATASET_TYPE} datasets on day_obs={DAY_OBS}")

FRAME_COLS = ("OCS", "CCS", "NW")
rows, agg_noll, shown = [], None, False
for ref in tqdm(refs, desc="Butler aggregate"):
    t = butler.get(ref)
    if not shown:                                # report the schema of the live table once
        print("aggregate columns:", list(t.colnames))
        print("meta keys:", list(t.meta))
        print("detectors present:", sorted(set(str(x) for x in t["detector"])))
        shown = True
    if agg_noll is None and t.meta.get("nollIndices") is not None:
        agg_noll = [int(x) for x in t.meta["nollIndices"]]
    det_raw = np.array([str(x) for x in t["detector"]])
    ids = np.array([NAME2ID.get(d, int(d) if d.lstrip("-").isdigit() else -1)
                    for d in det_raw])
    keep = np.isin(ids, list(CWFS))                 # rows are the SW0 (extra) corners
    if not keep.any():
        continue
    sub = t[keep]
    # ts_wep evaluates the intrinsic at each stamp's own CCS position; the aggregate
    # carries thx_CCS_intra/extra + thy_CCS_intra/extra, and rtp in meta['rotTelPos'].
    zkI = {f: np.asarray(sub[f"zk_intrinsic_{f}"], float)
           for f in FRAME_COLS if f"zk_intrinsic_{f}" in t.colnames}
    # meta['rotTelPos'] is in RADIANS -> degrees, wrapped to (-180, 180] (physical ~+-90)
    rtp_deg = (float(np.degrees(t.meta["rotTelPos"])) + 180.0) % 360.0 - 180.0
    rows.append(dict(
        visit=int(ref.dataId["visit"]),
        seq=int(ref.dataId["visit"] - DAY_OBS * 100000),
        det_id=ids[keep],
        rtp=rtp_deg,
        thx_CCS_intra=to_deg(sub["thx_CCS_intra"]), thy_CCS_intra=to_deg(sub["thy_CCS_intra"]),
        thx_CCS_extra=to_deg(sub["thx_CCS_extra"]), thy_CCS_extra=to_deg(sub["thy_CCS_extra"]),
        thx_CCS=to_deg(sub["thx_CCS"]), thy_CCS=to_deg(sub["thy_CCS"]),   # pair mean, for plots
        zkI=zkI))

assert rows, "no CWFS rows found -- check DATASET_TYPE / DAY_OBS / detector names"
frames = [f for f in FRAME_COLS if all(f in r["zkI"] for r in rows)]
print(f"{len(rows)} visits with CWFS rows; aggregate Noll (n={len(agg_noll)}): {agg_noll}")
print(f"intrinsic frames present: {frames}; rtp range "
      f"{min(r['rtp'] for r in rows):.1f}..{max(r['rtp'] for r in rows):.1f} deg")

# rotTelPos per seq_num (meta['rotTelPos']) -- check against expectations
print("\nrotTelPos per seq_num:")
for r in sorted(rows, key=lambda x: x["seq"]):
    print(f"  seq {r['seq']:>5d}  (visit {r['visit']})  rotTelPos = {r['rtp']:+8.3f} deg")

<a id='recompute'></a>
## 6. Recompute expected intrinsic (ts_wep recipe)

Reproduces `ts_wep` `calcZernikesTask`: for each pair, evaluate `getIntrinsicZernikes(field_x, field_y, rotTelPos)` at the **extra** stamp position (SW0 calib) and the **intra** stamp position (SW1 calib), then **average** the two — `rtp` from `meta['rotTelPos']` (radians -> degrees). The field-arg order, rtp sign, and comparison frame are taken from the Parameters (`FIELD_ORDER`, `RTP_SIGN`, `COMPARE_FRAME`) — not auto-solved. A per-frame `mean|corr|` is printed as a hint, but a mismatch will simply show in the plots.

In [ ]:
# Common Noll between calib and aggregate.
common = [j for j in agg_noll if j in calib_noll]
ai = [agg_noll.index(j) for j in common]        # -> aggregate zk_intrinsic columns
print(f"comparing {len(common)} shared Noll: {common}")


def eval_visit(r, fxb, fyb, rot):
    """ts_wep recipe: intrinsic at the EXTRA stamp position (SW0 calib) and the
    INTRA stamp position (SW1=SW0+1 calib), then averaged.  (n, len(common))."""
    n = len(r["det_id"])
    Ie = np.full((n, len(common)), np.nan)
    Ii = np.full((n, len(common)), np.nan)
    for de in np.unique(r["det_id"]):
        m = r["det_id"] == de
        di = int(de) + 1                            # SW1 (intra) partner
        Ie[m] = np.atleast_2d(calib_by_det[int(de)].getIntrinsicZernikes(
            field_x=np.asarray(r[fxb + "_extra"][m], float),
            field_y=np.asarray(r[fyb + "_extra"][m], float),
            rotTelPos=float(rot), noll_indices=common))
        if di in calib_by_det:
            Ii[m] = np.atleast_2d(calib_by_det[di].getIntrinsicZernikes(
                field_x=np.asarray(r[fxb + "_intra"][m], float),
                field_y=np.asarray(r[fyb + "_intra"][m], float),
                rotTelPos=float(rot), noll_indices=common))
        else:
            Ii[m] = Ie[m]                           # SW1 calib absent -> extra-only
    return 0.5 * (Ie + Ii)


# ---- fixed conventions from Parameters (no auto-solve) ----
FXB, FYB = FIELD_ORDER
FRAME = COMPARE_FRAME
print(f"conventions: field=({FXB},{FYB}), rtp sign={RTP_SIGN:+d}, compare-frame={FRAME}")

# hint only: mean|corr| of the expected vs EACH aggregate frame on a sample, so a
# wrong COMPARE_FRAME is obvious without changing what is used.
_smp = rows[:min(6, len(rows))]
_E = np.vstack([eval_visit(r, FXB, FYB, RTP_SIGN * r["rtp"]) for r in _smp])
for f in frames:
    _A = np.vstack([r["zkI"][f][:, ai] for r in _smp])
    cs = []
    for p in range(len(common)):
        x, y = _E[:, p], _A[:, p]
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() > 3 and np.std(x[m]) > 0 and np.std(y[m]) > 0:
            cs.append(abs(np.corrcoef(x[m], y[m])[0, 1]))
    print(f"  vs {f}: mean|corr| = {np.mean(cs) if cs else float('nan'):.3f}"
          + ("   <- COMPARE_FRAME" if f == FRAME else ""))

# ---- apply to ALL visits -> tidy comparison ----
recs = []
for r in rows:
    E = eval_visit(r, FXB, FYB, RTP_SIGN * r["rtp"])
    A = r["zkI"][FRAME][:, ai]
    for k, det in enumerate(r["det_id"]):
        for p, j in enumerate(common):
            recs.append(dict(visit=r["visit"], corner=CWFS[int(det)], j=int(j),
                             expected=E[k, p], aggregate=A[k, p], rtp=r["rtp"]))
cmp = pd.DataFrame(recs)
cmp["resid"] = cmp["expected"] - cmp["aggregate"]   # bracket: .aggregate is a DataFrame method
Path(OUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
cmp.to_parquet(OUT_PARQUET)

fin = np.isfinite(cmp["expected"]) & np.isfinite(cmp["aggregate"])
sl, off = np.polyfit(cmp["aggregate"][fin], cmp["expected"][fin], 1)
print(f"expected = {sl:.4f} * aggregate {off:+.4f}   (slope~1, offset~0 => values match; "
      f"slope~1e6/1e-6 => a unit mismatch)")
print(f"wrote {OUT_PARQUET}  ({len(cmp)} rows)")

# ---- MIW calib sanity: are the recomputed intrinsics FINITE and reasonable? ----
# (This is the useful check on data where the MIW was NOT applied, e.g.
#  nightlyValidation -- the PASS/FAIL vs the aggregate is only meaningful when
#  THIS MIW was the one applied.)
exp = cmp["expected"].to_numpy(float)
nan_frac = float(np.mean(~np.isfinite(exp)))
fin = exp[np.isfinite(exp)]
print(f"\nMIW calib sanity: {len(exp)} expected values, {100 * nan_frac:.1f}% NaN")
if fin.size:
    print(f"  |expected| median {np.median(np.abs(fin)):.3f}, "
          f"95pct {np.percentile(np.abs(fin), 95):.3f}, max {np.max(np.abs(fin)):.3f} µm "
          f"(intrinsic Zernikes should be ~sub-µm to a few µm)")
nan_by_corner = (cmp.assign(bad=~np.isfinite(cmp["expected"]))
                    .groupby("corner")["bad"].mean())
print("  NaN fraction by corner:", {c: round(float(v), 3) for c, v in nan_by_corner.items()})
if nan_frac > 0:
    nan_by_j = (cmp.assign(bad=~np.isfinite(cmp["expected"]))
                   .groupby("j")["bad"].mean())
    bad_j = [int(j) for j, v in nan_by_j.items() if v > 0]
    print(f"  WARNING: MIW returned NaN at some CWFS positions (Noll {bad_j}). Likely the "
          f"(rotated) field point fell outside the calib interpolation hull -- corners near "
          f"the 1.75 deg edge, or a missing Noll in the OCS/CCS tables.")
else:
    print("  OK: MIW returned finite values at every CWFS position.")

print("\nNOTE: on nightlyValidation the MIW was NOT applied, so the PASS/FAIL vs the "
      "aggregate's zk_intrinsic is EXPECTED to fail; the sanity result above (finite, "
      "sub-µm) is the meaningful check. Run against data where this MIW was applied to "
      "make the comparison meaningful.")

<a id='compare'></a>
## 7. Comparison plots + PASS/FAIL

In [ ]:
# ---- PASS/FAIL: max median |residual| over corner x Noll ----
gg = (cmp.dropna(subset=["resid"]).groupby(["corner", "j"]).resid
        .apply(lambda x: float(np.median(np.abs(x)))))
max_med = float(gg.max())
print(f"max median|expected-aggregate| over corner x Noll = {max_med:.4f} µm  "
      f"(TOL={TOL_UM})  ->  {'PASS' if max_med <= TOL_UM else 'FAIL'}")
print("\nworst (corner, Noll):")
print(gg.sort_values(ascending=False).head(10).to_string())

# ---- one scatter panel PER ZERNIKE term (corners kept as colors) ----
jvals = sorted(cmp.j.unique())
corners = sorted(cmp.corner.unique())
cmapc = {c: plt.cm.tab10(i % 10) for i, c in enumerate(corners)}
ncol = 5; nrow = int(np.ceil(len(jvals) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.1 * nrow),
                        layout="constrained", squeeze=False)
for ax, j in zip(axs.ravel(), jvals):
    d = cmp[cmp.j == j].dropna(subset=["expected", "aggregate"])
    for c in corners:
        dc = d[d.corner == c]
        ax.scatter(dc["aggregate"], dc["expected"], s=14, alpha=0.65,
                   color=cmapc[c], label=c)
    if len(d):
        lo = float(min(d["aggregate"].min(), d["expected"].min()))
        hi = float(max(d["aggregate"].max(), d["expected"].max()))
        pad = 0.05 * (hi - lo + 1e-9)
        ax.plot([lo, hi], [lo, hi], "k:", lw=0.8)
        ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
        rms = float(np.sqrt(np.nanmean((d["expected"] - d["aggregate"]) ** 2)))
        ax.set_title(f"Z{j}   rms={rms:.4f} µm", fontsize=9)
    else:
        ax.set_title(f"Z{j}", fontsize=9)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("aggregate [µm]", fontsize=8); ax.set_ylabel("expected [µm]", fontsize=8)
    ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
for ax in axs.ravel()[len(jvals):]:
    ax.axis("off")
axs.ravel()[0].legend(fontsize=6, loc="best")
fig.suptitle(f"CWFS intrinsic: expected (calib) vs aggregate, per Zernike — "
             f"day {DAY_OBS} [frame {FRAME}]", fontsize=12)
plt.show()

# ---- heatmap: median |residual| per (corner, Noll) ----
piv = (cmp.dropna(subset=["resid"]).assign(absr=lambda d: d.resid.abs())
         .pivot_table(index="corner", columns="j", values="absr", aggfunc="median"))
fig, ax = plt.subplots(figsize=(1.1 + 0.45 * piv.shape[1], 1.0 + 0.45 * piv.shape[0]),
                       layout="constrained")
im = ax.imshow(piv.values, aspect="auto", cmap="magma_r",
               vmin=0, vmax=max(TOL_UM, float(np.nanpercentile(piv.values, 95))))
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"Z{j}" for j in piv.columns], fontsize=7)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
fig.colorbar(im, ax=ax, label="median |resid| [µm]")
ax.set_title(f"median |expected-aggregate| (TOL={TOL_UM} µm)", fontsize=10)
plt.show()

<a id='nandiag'></a>\n## 8. NaN diagnosis -- OCS hull vs CCS footprint\n\nPer corner, where (in field) does the MIW return NaN? Attribute each donut's NaN to the rotated OCS interpolator vs the per-detector CCS interpolator, and overlay the donut positions on the OCS sample coverage and the CCS footprints. If the NaN donuts sit **outside** the OCS samples (gray), it's the MIW map's data hull not reaching the corner positions; if outside the CCS footprint (green), it's the per-detector calib.

In [ ]:
# Attribute each donut's NaN to the OCS interpolator (rotated) vs the per-detector
# CCS interpolator vs neither, then map donut positions (finite/NaN) over the
# OCS sample coverage (gray) and the CCS footprints (green), per corner.
def _rot2d(x, y, deg):
    a = np.radians(deg); c, s = np.cos(a), np.sin(a)
    return c * x - s * y, s * x + c * y

recs = []
for r in rows:
    rot = RTP_SIGN * r["rtp"]
    for side in ("extra", "intra"):
        xs = np.asarray(r[f"thx_CCS_{side}"], float)
        ys = np.asarray(r[f"thy_CCS_{side}"], float)
        dets = np.asarray(r["det_id"]) + (0 if side == "extra" else 1)   # SW0=extra, SW1=intra
        for dd in np.unique(dets):
            m = dets == dd; xx, yy = xs[m], ys[m]
            cal = calib_by_det.get(int(dd))
            if cal is None:
                fn = cn = on = onr = np.ones(int(m.sum()), bool)
            else:
                full = np.atleast_2d(cal.getIntrinsicZernikes(
                    field_x=xx, field_y=yy, rotTelPos=float(rot), noll_indices=common))
                fn = np.isnan(full).any(1)
                cn = np.isnan(np.atleast_2d(cal.interpolator(np.column_stack([xx, yy])))).any(1)
                on = np.isnan(np.atleast_2d(cal.interpolator_ocs(np.column_stack([xx, yy])))).any(1)
                xr, yr = _rot2d(xx, yy, rot)
                onr = np.isnan(np.atleast_2d(cal.interpolator_ocs(np.column_stack([xr, yr])))).any(1)
            base = int(dd) - (0 if side == "extra" else 1)
            for i in range(int(m.sum())):
                recs.append(dict(corner=CWFS.get(base), side=side, det=int(dd),
                                 x=float(xx[i]), y=float(yy[i]),
                                 full_nan=bool(fn[i]), ccs_nan=bool(cn[i]),
                                 ocs_nan=bool(on[i]), ocs_nan_rot=bool(onr[i])))
diag = pd.DataFrame(recs)
print(f"donut evaluations: {len(diag)}   full NaN {diag.full_nan.mean():.1%}")
print(f"  CCS(unrot) NaN {diag.ccs_nan.mean():.1%} | OCS(unrot) NaN {diag.ocs_nan.mean():.1%} "
      f"| OCS(rot {RTP_SIGN:+d}) NaN {diag.ocs_nan_rot.mean():.1%}")
for src in ("ccs_nan", "ocs_nan", "ocs_nan_rot"):
    cap = (diag.full_nan & diag[src]).sum()
    print(f"  full_nan agrees with {src}: {(diag.full_nan == diag[src]).mean():.1%} "
          f"(captures {cap}/{int(diag.full_nan.sum())} of the NaNs)")

# --- per-corner position maps ---
ocsx, ocsy = np.asarray(cal0.field_x_ocs), np.asarray(cal0.field_y_ocs)
corners = [CWFS[d] for d in sorted(CWFS)]
fig, axes = plt.subplots(1, len(corners), figsize=(4.4 * len(corners), 4.6),
                         constrained_layout=True, squeeze=False)
for ax, cn in zip(axes.ravel(), corners):
    d = diag[diag.corner == cn]
    dx, dy, fn = d.x.to_numpy(), d.y.to_numpy(), d.full_nan.to_numpy()
    ax.scatter(ocsx, ocsy, s=1, color="0.8", label="OCS samples")
    de = NAME2ID[cn]
    for dd in (de, de + 1):
        cal = calib_by_det.get(dd)
        if cal is not None:
            ax.scatter(np.asarray(cal.field_x), np.asarray(cal.field_y), s=3, color="green", alpha=0.5)
    ax.scatter(dx[~fn], dy[~fn], s=6, color="steelblue", label="finite")
    ax.scatter(dx[fn], dy[fn], s=6, color="crimson", label="NaN")
    ax.add_patch(plt.Circle((0, 0), FP_RADIUS_DEG, fill=False, ls="--", color="k", lw=0.5))
    ax.set_aspect("equal"); ax.set_title(f"{cn}  NaN {fn.mean():.0%}", fontsize=10)
    ax.set_xlabel("thx_CCS [deg]"); ax.set_ylabel("thy_CCS [deg]"); ax.legend(fontsize=6)
fig.suptitle("MIW NaN by donut position -- OCS samples (gray), CCS footprint (green), "
             "donut finite (blue) / NaN (red)", fontsize=12)
plt.show()

# --- zoom on the first corner: donuts vs the OCS-sample edge + CCS footprint ---
cn = corners[0]; d = diag[diag.corner == cn]
dx, dy, fn = d.x.to_numpy(), d.y.to_numpy(), d.full_nan.to_numpy()
fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
ax.scatter(ocsx, ocsy, s=4, color="0.8", label="OCS samples")
de = NAME2ID[cn]
for dd in (de, de + 1):
    cal = calib_by_det.get(dd)
    if cal is not None:
        ax.scatter(np.asarray(cal.field_x), np.asarray(cal.field_y), s=8, color="green",
                   alpha=0.6, label=f"CCS footprint det{dd}")
ax.scatter(dx[~fn], dy[~fn], s=12, color="steelblue", label="finite")
ax.scatter(dx[fn], dy[fn], s=12, color="crimson", label="NaN")
ax.set_xlim(np.nanmedian(dx) - 0.6, np.nanmedian(dx) + 0.6)
ax.set_ylim(np.nanmedian(dy) - 0.6, np.nanmedian(dy) + 0.6)
ax.set_aspect("equal"); ax.legend(fontsize=7)
ax.set_title(f"{cn}: zoom -- donut positions vs OCS/CCS coverage")
plt.show()